# WR Churn Risk Prediction Using a Synthetic Mobile MOBA Dataset

This notebook evaluates churn-risk modeling using a synthetic dataset designed to approximate player lifecycle patterns in a fast-paced mobile MOBA environment.

The workflow compares Random Forest, XGBoost, and LightGBM for early retention risk prediction. The primary use case is identifying players with elevated short-term churn risk, while a secondary use case is supporting reactivation targeting.

This project is intended for portfolio demonstration only. The dataset is synthetic and does not reproduce internal production logic.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import shap

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)
plt.rcParams["figure.figsize"] = (10, 6)

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
DATA_PATH = "data/synthetic_wr_dataset.csv"
TARGET = "churn_risk"
ID_COL = "player_id"
MATCH_COL = "matches_last_7d"
RANDOM_STATE = 42

## 1. Problem Statement

Player disengagement is a major retention challenge in competitive mobile games, especially during the early lifecycle. In a fast-paced mobile MOBA setting, behavioral signals such as recent match volume, activity consistency, progression, social play, and friction indicators can provide early warning signs of elevated churn risk.

This notebook demonstrates a public churn-risk modeling workflow using a synthetic Wild Rift-inspired dataset. The target variable represents short-term retention risk rather than an observed production churn event. The objective is to compare tree-based models and identify a practical approach for early retention monitoring.

## 2. Load Data

The dataset used here is synthetic and designed for portfolio-safe demonstration. Each row represents one player over a trailing 7-day behavioral window.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.shape, df.columns.tolist()

## 3. Target Definition

The target variable `churn_risk` is a synthetic short-term retention risk label generated from recency, activity, progression, social play, and friction-related signals. A positive label indicates elevated near-term churn risk.

In [ ]:
print(df[TARGET].value_counts(dropna=False))
print(df[TARGET].value_counts(normalize=True, dropna=False).round(3))

## 4. Data Overview

This section checks schema quality, missing values, and basic feature composition before modeling.

In [ ]:
df.info()

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0]

In [ ]:
categorical_cols = df.drop(columns=[TARGET, ID_COL]).select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = df.drop(columns=[TARGET, ID_COL]).select_dtypes(include=["number"]).columns.tolist()

print("Categorical columns:", len(categorical_cols))
print("Numeric columns:", len(numeric_cols))
print("\nCategorical:", categorical_cols)
print("\nNumeric:", numeric_cols)

## 5. Exploratory Checks

A few quick checks help validate whether the synthetic dataset reflects sensible lifecycle patterns.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df[MATCH_COL], bins=20, kde=False, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of Matches in Last 7 Days")
axes[0].set_xlabel("matches_last_7d")

sns.histplot(df["last_login_gap_days"], bins=15, kde=False, ax=axes[1], color="darkorange")
axes[1].set_title("Distribution of Last Login Gap")
axes[1].set_xlabel("last_login_gap_days")

sns.countplot(data=df, x=TARGET, ax=axes[2], palette=["steelblue", "orange"])
axes[2].set_title("Churn Risk Class Balance")
axes[2].set_xlabel("churn_risk")
axes[2].set_ylabel("count")

plt.tight_layout()
plt.show()

In [ ]:
risk_by_region = df.groupby("region")[TARGET].mean().sort_values(ascending=False).round(3)
risk_by_region

## 6. Prepare Features

The identifier column is excluded from model input. Numeric features are median-imputed, while categorical features are imputed with the most frequent value and one-hot encoded.

In [ ]:
X = df.drop(columns=[TARGET, ID_COL])
y = df[TARGET]

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

## 7. Model Selection

Three tree-based models are compared:

- **Random Forest** as a strong and interpretable baseline
- **XGBoost** as a high-performing boosted-tree benchmark
- **LightGBM** as an efficient gradient boosting model suitable for tabular retention-risk tasks

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

## 8. Cross-Validation Model Comparison

Model performance is evaluated with stratified 10-fold cross-validation using accuracy, precision, recall, F1-score, and AUC.

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "auc": "roc_auc"
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "AUC": scores["test_auc"].mean()
    })

results_df = pd.DataFrame(results).sort_values("Recall", ascending=False)
results_df.round(3)

## 9. Model Performance Visualization

In [ ]:
plot_df = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1", "AUC"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(11, 6))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric")
plt.ylim(0.5, 1.0)
plt.title("Model Performance Comparison Across Key Metrics")
plt.ylabel("Score")
plt.tight_layout()
plt.show()

## 10. Final LightGBM Evaluation

LightGBM is selected for deeper inspection because it typically offers a strong balance between recall, efficiency, and practical suitability for churn-risk monitoring in tabular settings.

In [ ]:
lightgbm_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", models["LightGBM"])
])

pred = cross_val_predict(lightgbm_pipe, X, y, cv=cv, method="predict", n_jobs=-1)
proba = cross_val_predict(lightgbm_pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

metrics_summary = {
    "Accuracy": accuracy_score(y, pred),
    "Precision": precision_score(y, pred),
    "Recall": recall_score(y, pred),
    "F1": f1_score(y, pred),
    "AUC": roc_auc_score(y, proba)
}

pd.Series(metrics_summary).round(3)

In [ ]:
print(classification_report(y, pred, target_names=["Lower Risk", "Higher Risk"]))

In [ ]:
cm = confusion_matrix(y, pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Lower Risk", "Higher Risk"],
    yticklabels=["Lower Risk", "Higher Risk"]
)
plt.title("Confusion Matrix for LightGBM Churn Risk Prediction")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 11. Churn Risk by Recent Match Volume

This view illustrates how predicted risk is distributed across different levels of recent match activity.

In [ ]:
tmp = df[[MATCH_COL, TARGET]].copy()
tmp[MATCH_COL] = tmp[MATCH_COL].clip(upper=20)

summary = (
    tmp.groupby([MATCH_COL, TARGET])
       .size()
       .reset_index(name="count")
)
summary["pct"] = summary.groupby(MATCH_COL)["count"].transform(lambda s: 100 * s / s.sum())

pivot = summary.pivot(index=MATCH_COL, columns=TARGET, values="pct").fillna(0)

if 0 in pivot.columns and 1 in pivot.columns:
    pivot = pivot.rename(columns={0: "Lower Risk", 1: "Higher Risk"})

pivot[[col for col in ["Higher Risk", "Lower Risk"] if col in pivot.columns]].plot(
    kind="bar", figsize=(10, 5), color=["orange", "steelblue"]
)
plt.title("Churn Risk by Recent Match Volume")
plt.xlabel("Matches in Last 7 Days")
plt.ylabel("Percentage of Players")
plt.tight_layout()
plt.show()

## 12. SHAP Explainability

SHAP is used to inspect which features contribute most strongly to elevated churn-risk predictions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

lightgbm_pipe.fit(X_train, y_train)

X_test_processed = lightgbm_pipe.named_steps["prep"].transform(X_test)
feature_names = lightgbm_pipe.named_steps["prep"].get_feature_names_out()
lightgbm_model = lightgbm_pipe.named_steps["model"]

sample_size = min(1000, X_test_processed.shape[0])
sample_idx = np.random.RandomState(RANDOM_STATE).choice(X_test_processed.shape[0], size=sample_size, replace=False)
X_shap = X_test_processed[sample_idx]

if hasattr(X_shap, "toarray"):
    X_shap = X_shap.toarray()

X_shap_df = pd.DataFrame(X_shap, columns=feature_names)

explainer = shap.TreeExplainer(lightgbm_model)
shap_values = explainer.shap_values(X_shap_df)

if isinstance(shap_values, list):
    shap_values_to_plot = shap_values[1]
else:
    shap_values_to_plot = shap_values

In [ ]:
shap.summary_plot(shap_values_to_plot, X_shap_df, show=False)
plt.tight_layout()
plt.show()

## 13. Lifecycle Risk Heatmap

This conceptual heatmap summarizes where common data and deployment risks can surface across the churn modeling lifecycle.

In [ ]:
risk_df = pd.DataFrame({
    "Collection": [2, 3, 2, 0, 3],
    "Preprocessing": [2, 2, 3, 0, 2],
    "Model Training": [1, 3, 1, 2, 3],
    "Inference": [1, 2, 1, 3, 2],
    "Deployment": [1, 2, 1, 3, 3]
}, index=["Missing Data", "Bias", "Time Inconsistency", "Explainability Risk", "Cold Start"])

plt.figure(figsize=(8, 5))
sns.heatmap(risk_df, annot=True, cmap="YlOrRd", cbar=True)
plt.title("Data Risk Heatmap Across Lifecycle Stages")
plt.tight_layout()
plt.show()

## 14. Key Findings

- Boosted tree models are typically stronger than the Random Forest baseline in this synthetic churn-risk setting.
- Recall is especially important because missing a higher-risk player is often more costly than flagging a manageable number of lower-risk players.
- Recent activity, login recency, social play, progression, and friction-related signals are all useful for identifying elevated retention risk.
- LightGBM is a practical candidate for deeper inspection because it balances predictive performance and efficiency well for tabular workflows.
- This notebook should be interpreted as a portfolio-safe churn-risk modeling demonstration rather than a production churn system.

## 15. Business Implications

This workflow can support two practical retention use cases:

1. **Early retention monitoring**
   - flag newer or declining players for low-cost intervention
   - prioritize cohorts with low recent activity and widening login gaps

2. **Reactivation targeting**
   - identify lapsed players with enough historical engagement to justify outreach
   - combine churn-risk output with messaging, event, or reward strategies

In a real production environment, the next step would be threshold tuning, intervention design, and online validation against real retention outcomes.

## 16. Reproducibility Note

Suggested environment:
- Python 3.10+
- pandas
- numpy
- matplotlib
- seaborn
- scikit-learn
- xgboost
- lightgbm
- shap

Run all cells before publishing the notebook so GitHub displays the tables and charts directly.